In [64]:
import pandas as pd
import numpy as np

In [65]:
data = pd.read_csv("../bitcoin_data/raw/btc_3h.csv")
data.head()

,datetime,open,high,low,close,volume
0,2020-01-01 00:00:00,7195.24,7244.87,7175.46,7242.85,2050.024313
1,2020-01-01 03:00:00,7242.66,7245.00,7215.03,7224.21,1596.208041
2,2020-01-01 06:00:00,7224.24,7236.27,7180.00,7200.64,2164.357851
3,2020-01-01 09:00:00,7200.29,7237.73,7185.20,7197.20,2251.108387
4,2020-01-01 12:00:00,7197.20,7234.97,7196.15,7221.43,2098.648667


In [66]:
data.tail()

,datetime,open,high,low,close,volume
18656,2026-05-21 09:00:00,77924.63,77977.53,77147.15,77189.10,1441.40032
18657,2026-05-21 12:00:00,77189.10,77363.32,76719.47,76984.61,1942.79991
18658,2026-05-21 15:00:00,76984.61,78098.16,76900.00,77869.38,2107.40302
18659,2026-05-21 18:00:00,77869.38,77922.01,77230.00,77705.81,1141.78197
18660,2026-05-21 21:00:00,77705.80,77813.35,77705.80,77780.37,101.88650


In [67]:
df = data.copy()

##### Feature Engineering

In [68]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = -delta.clip(upper=0).rolling(period).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))


In [69]:
def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast   = series.ewm(span=fast, adjust=False).mean()
    ema_slow   = series.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line= macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return macd_line, signal_line, histogram


In [70]:
def add_features(df):
    close = df['close'].squeeze()
    high  = df['high'].squeeze()
    low   = df['low'].squeeze()
    vol   = df['volume'].squeeze()
    open_ = df['open'].squeeze()
    


     # === Trend Indicators ===
    df['ma5']   = close.rolling(5).mean()
    df['ma10']  = close.rolling(10).mean()
    df['ma20']  = close.rolling(20).mean()
    df['ma50']  = close.rolling(50).mean()
    df['ema12'] = close.ewm(span=12, adjust=False).mean()
    df['ema26'] = close.ewm(span=26, adjust=False).mean()
 
    # MA Crossover signals
    df['ma5_above_ma20']  = (df['ma5'] > df['ma20']).astype(int)
    df['ema12_above_ema26'] = (df['ema12'] > df['ema26']).astype(int)
 
    # === Momentum ===
    df['rsi14'] = compute_rsi(close, 14)
    df['rsi7']  = compute_rsi(close, 7)
    df['macd'], df['macd_signal'], df['macd_hist'] = compute_macd(close)
    df['momentum5']  = close - close.shift(5)
    df['momentum10'] = close - close.shift(10)
 
    # === Volatility ===
    df['bb_mid']   = close.rolling(20).mean()
    bb_std         = close.rolling(20).std()
    df['bb_upper'] = df['bb_mid'] + 2 * bb_std
    df['bb_lower'] = df['bb_mid'] - 2 * bb_std
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']
    df['bb_pos']   = (close - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    df['atr14']    = (high - low).rolling(14).mean()  # simplified ATR
 
    # === Volume ===
    df['vol_change']  = vol.pct_change()
    df['vol_ma5']     = vol.rolling(5).mean()
    df['vol_ratio']   = vol / df['vol_ma5']
 
    # === Price Action ===
    df['high_low_range']   = (high - low) / close
    df['close_open_diff']  = (close - open_) / open_
    df['upper_shadow']     = (high - pd.concat([close, open_], axis=1).max(axis=1)) / close
    df['lower_shadow']     = (pd.concat([close, open_], axis=1).min(axis=1) - low) / close
 
    # === Lag Features (গত দিনের তথ্য) ===
    for lag in [1, 2, 3, 5]:
        df[f'close_lag{lag}'] = close.pct_change(lag)
        df[f'vol_lag{lag}']   = vol.pct_change(lag)
 
    # === TARGET ===
    df['target'] = close.pct_change().shift(-1) * 100  # পরের দিনের % change
 
    df.dropna(inplace=True)
    return df



In [71]:
# ─── 3. Feature List ─────────────────────────────────────────────────
feature_cols = [
    'ma5', 'ma10', 'ma20', 'ma50', 'ema12', 'ema26',
    'ma5_above_ma20', 'ema12_above_ema26',
    'rsi14', 'rsi7',
    'macd', 'macd_signal', 'macd_hist',
    'momentum5', 'momentum10',
    'bb_width', 'bb_pos', 'atr14',
    'vol_change', 'vol_ratio',
    'high_low_range', 'close_open_diff',
    'upper_shadow', 'lower_shadow',
    'close_lag1', 'close_lag2', 'close_lag3', 'close_lag5',
    'vol_lag1', 'vol_lag2', 'vol_lag3', 'vol_lag5',
]


In [72]:
df = add_features(df)
print(f"✅ Shape: {df.shape}")
print(f"✅ Target sample:\n{df['target'].describe()}")


✅ Shape: (18611, 43)
✅ Target sample:
count    18611.000000
mean         0.018446
std          1.106063
min        -17.764895
25%         -0.397280
50%          0.013788
75%          0.431919
max         10.558770
Name: target, dtype: float64


In [73]:
df['target'] = df['close'].pct_change().shift(-1) * 100
df.dropna(subset=['target'], inplace=True)

print(df['target'].describe())
print(f"\nPositive (দাম বাড়বে): {(df['target'] > 0).sum()} ({(df['target'] > 0).mean()*100:.1f}%)")
print(f"Negative (দাম কমবে): {(df['target'] < 0).sum()} ({(df['target'] < 0).mean()*100:.1f}%)")

count    18610.000000
mean         0.018442
std          1.106092
min        -17.764895
25%         -0.397325
50%          0.013710
75%          0.431953
max         10.558770
Name: target, dtype: float64

Positive (দাম বাড়বে): 9511 (51.1%)
Negative (দাম কমবে): 9099 (48.9%)


In [74]:
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime")
df = df.set_index("datetime")

In [75]:
split_date = "2025-01-01"

train = df[df.index < split_date]
test = df[df.index >= split_date]
print(f"Train: {len(train)} rows | Test: {len(test)} rows")


Train: 14564 rows | Test: 4046 rows


In [76]:
# Step 3: X, y separate
X_train = train[feature_cols]
y_train = train['target']
X_test  = test[feature_cols]
y_test  = test['target']


In [77]:
# Step 4: Scale করুন
from sklearn.preprocessing import StandardScaler
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n✅ X_train: {X_train_sc.shape}")
print(f"✅ X_test : {X_test_sc.shape}")



✅ X_train: (14564, 32)
✅ X_test : (4046, 32)


In [79]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train_sc, y_train)
preds = xgb.predict(X_test_sc)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print(f"MAE  : {mae:.4f}%")
print(f"RMSE : {rmse:.4f}%")
print(f"R²   : {r2:.4f}")

MAE  : 0.5688%
RMSE : 0.8582%
R²   : -0.0551


In [81]:
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)

params_list = [
    {'n_estimators': 300,  'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8},
    {'n_estimators': 500,  'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8},
    {'n_estimators': 1000, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.7},
    {'n_estimators': 500,  'max_depth': 2, 'learning_rate': 0.05, 'subsample': 0.9},
]

best_mae   = np.inf
best_param = None

for params in params_list:
    maes = []
    for train_idx, val_idx in tscv.split(X_train_sc):
        X_tr, X_val = X_train_sc[train_idx], X_train_sc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        m = XGBRegressor(**params, random_state=42, verbosity=0)
        m.fit(X_tr, y_tr)
        maes.append(mean_absolute_error(y_val, m.predict(X_val)))

    avg_mae = np.mean(maes)
    print(f"MAE: {avg_mae:.4f} | {params}")

    if avg_mae < best_mae:
        best_mae   = avg_mae
        best_param = params

print(f"\n✅ Best params: {best_param}")
print(f"✅ Best MAE   : {best_mae:.4f}%")

MAE: 0.8116 | {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8}
MAE: 0.8982 | {'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8}
MAE: 0.8636 | {'n_estimators': 1000, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.7}
MAE: 0.9308 | {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.05, 'subsample': 0.9}

✅ Best params: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8}
✅ Best MAE   : 0.8116%


In [82]:
final_xgb = XGBRegressor(**best_param, random_state=42, verbosity=0)
final_xgb.fit(X_train_sc, y_train)
final_preds = final_xgb.predict(X_test_sc)

print(f"MAE  : {mean_absolute_error(y_test, final_preds):.4f}%")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, final_preds)):.4f}%")
print(f"R²   : {r2_score(y_test, final_preds):.4f}")

MAE  : 0.5484%
RMSE : 0.8377%
R²   : -0.0052
